# 01 - Deploy BYOM Resource Foundation

Complete [00_setup.ipynb](./00_setup.ipynb) and select **Python (byom01-capability-testing)**.

## Deployment Route

Use the section that matches your resources.

### Deploy via Notebooks

Run this notebook to provision the demo, then [02_configure.ipynb](./02_configure.ipynb) to configure the gateway and [03_test.ipynb](./03_test.ipynb) to verify it. You can then run [04_foundry_tools.ipynb](./04_foundry_tools.ipynb) or [05_direct_api.ipynb](./05_direct_api.ipynb) with the same venv and generated settings. This deployment uses **Microsoft Entra ID**.

### Self deployed

If you already deployed your Azure resources independently, **skip the code in this notebook and notebooks 02/03**. Open [04_foundry_tools.ipynb](./04_foundry_tools.ipynb) or [05_direct_api.ipynb](./05_direct_api.ipynb), keep the venv from Setup, and supply the environment variables listed there.

---

## What This Notebook Deploys

Provision APIM **Standard**, a consuming Foundry **S0** account/project, and Azure AI Search **S1**. Search is provisioned as part of the resource foundation; no search examples or indexes are configured.

A separate Foundry **S0** account hosts **gpt-5.4** in **Sweden Central**. The consuming resources default to **UK South**.

**APIM and Search incur charges even when idle.** This notebook provisions resources; Configure creates the gateway route, and Test makes a paid model request.

Bicep defines the infrastructure. Python SDK calls deploy the compiled template and save identifiers/endpoints to `env/.env`. No keys are retrieved.

## Configuration

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

DEMO_ID = "byom01"
RESOURCE_GROUP = "rg-byom01-uks"
LOCATION = "uksouth"
DEPLOYMENT_NAME = "main-byom01"
BACKEND_RESOURCE_GROUP = "rg-byom01-backend-swc"
BACKEND_LOCATION = "swedencentral"
BACKEND_DEPLOYMENT_NAME = "backend-byom01"
MODEL_DEPLOYMENT_NAME = "gpt-5.4"
MODEL_SKU = "GlobalStandard"
MODEL_CAPACITY = 100  # 100,000 tokens per minute
backend_outputs = None

demo_root = Path("..").resolve()
bicep_file = demo_root / "infra" / "main.bicep"
backend_bicep_file = demo_root / "infra" / "backend.bicep"
env_path = demo_root / "env" / ".env"
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found. Run the setup notebook first.")

print(f"Resource group: {RESOURCE_GROUP}")
print(f"Location: {LOCATION}")
print("Paid tiers: APIM Standard (1 unit), Foundry S0, Search S1 (1 replica, 1 partition)")
print(f"Backend: {BACKEND_RESOURCE_GROUP}, {BACKEND_LOCATION}, gpt-5.4 {MODEL_SKU}, capacity {MODEL_CAPACITY}")

## Resolve Subscription and Authentication

Use `AZURE_SUBSCRIPTION_ID` when set; otherwise use the Azure CLI subscription selected during setup. No subscription is selected from a list.

In [ ]:
from azure.identity import AzureCliCredential
from azure.mgmt.resource import ResourceManagementClient

account_cmd = [az_cmd, "account", "show", "--output", "json"]
requested_subscription = os.getenv("AZURE_SUBSCRIPTION_ID")
if requested_subscription:
    account_cmd.extend(["--subscription", requested_subscription])

account_result = subprocess.run(account_cmd, capture_output=True, text=True)
if account_result.returncode != 0:
    raise RuntimeError(
        f"az account show failed (exit {account_result.returncode})\n"
        f"stderr: {account_result.stderr.strip()}\n"
        f"stdout: {account_result.stdout.strip()}"
    )

account = json.loads(account_result.stdout)
subscription_id = account["id"]
credential = AzureCliCredential(tenant_id=account["tenantId"])
resource_client = ResourceManagementClient(credential, subscription_id)
print(f"Subscription: {account['name']} ({subscription_id})")

## Select Backend Model and Check Quota

Resolve the current generally available GPT-5.4 default version in Sweden Central. Check GlobalStandard quota before creating resources. No model, region, or SKU fallback is used.

**GlobalStandard can process requests outside Sweden.** This region selection is not a Sweden-only data-residency guarantee. Ensure subscription policy permits the public, Entra-authenticated backend.

In [ ]:
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient

cognitive_client = CognitiveServicesManagementClient(credential, subscription_id)
backend_model = next((
    entry.model for entry in cognitive_client.models.list(BACKEND_LOCATION)
    if entry.kind == "AIServices" and entry.model is not None
    and entry.model.name == "gpt-5.4" and entry.model.is_default_version
    and entry.model.lifecycle_status == "GenerallyAvailable"
), None)
if backend_model is None or not backend_model.version:
    raise RuntimeError(f"No generally available default gpt-5.4 version in {BACKEND_LOCATION}.")
backend_sku = next((sku for sku in backend_model.skus if sku.name == MODEL_SKU), None)
if backend_sku is None:
    raise RuntimeError(f"gpt-5.4 does not support {MODEL_SKU} in {BACKEND_LOCATION}.")
if backend_sku.deprecation_date is not None and backend_sku.deprecation_date <= datetime.now(timezone.utc):
    raise RuntimeError(f"gpt-5.4 {backend_model.version} {MODEL_SKU} is deprecated.")
quota = next(usage for usage in cognitive_client.usages.list(BACKEND_LOCATION)
             if usage.name.value == backend_sku.usage_name)
available_capacity = quota.limit - quota.current_value
if available_capacity < MODEL_CAPACITY:
    raise RuntimeError(f"Insufficient {MODEL_SKU} gpt-5.4 quota: {available_capacity} available, {MODEL_CAPACITY} requested.")
model_version = backend_model.version
print(f"Selected gpt-5.4 {model_version}, {MODEL_SKU}, capacity {MODEL_CAPACITY}")
print(f"Available quota: {available_capacity}")

## Compile Bicep

In [ ]:
build_result = subprocess.run(
    [az_cmd, "bicep", "build", "--file", str(bicep_file), "--stdout"],
    capture_output=True,
    text=True,
)
if build_result.returncode != 0:
    raise RuntimeError(
        f"az bicep build failed (exit {build_result.returncode})\n"
        f"stderr: {build_result.stderr.strip()}\n"
        f"stdout: {build_result.stdout.strip()}"
    )

template = json.loads(build_result.stdout)
print("Bicep compiled")

## Compile Backend Bicep

In [ ]:
backend_build_result = subprocess.run(
    [az_cmd, "bicep", "build", "--file", str(backend_bicep_file), "--stdout"],
    capture_output=True,
    text=True,
)
if backend_build_result.returncode != 0:
    raise RuntimeError(
        f"az bicep build failed (exit {backend_build_result.returncode})\n"
        f"stderr: {backend_build_result.stderr.strip()}\n"
        f"stdout: {backend_build_result.stdout.strip()}"
    )

backend_template = json.loads(backend_build_result.stdout)
print("Backend Bicep compiled")

## Create Resource Group

In [ ]:
resource_client.resource_groups.create_or_update(
    RESOURCE_GROUP,
    {"location": LOCATION},
)
print(f"Created or updated resource group: {RESOURCE_GROUP}")

## Deploy Bicep

Wait for completion; APIM provisioning can take a while. If deployment fails, follow the README's partial-deployment cleanup instructions. Do not run the environment-write step after a failure.

In [ ]:
outputs = None
print("Deploying infrastructure...")
deployment = resource_client.deployments.begin_create_or_update(
    RESOURCE_GROUP,
    DEPLOYMENT_NAME,
    {
        "properties": {
            "mode": "Incremental",
            "template": template,
            "parameters": {
                "demoId": {"value": DEMO_ID},
                "location": {"value": LOCATION},
            },
        }
    },
).result()
outputs = deployment.properties.outputs
print("Deployment completed")
print(json.dumps(outputs, indent=2))

## Create Backend Resource Group

In [ ]:
resource_client.resource_groups.create_or_update(
    BACKEND_RESOURCE_GROUP,
    {"location": BACKEND_LOCATION},
)
print(f"Created or updated backend resource group: {BACKEND_RESOURCE_GROUP}")

## Deploy Backend Bicep

Provision the separate Foundry account and GPT-5.4 deployment. Bicep disables local key authentication; no keys or runtime role assignments are created in this slice. Wait for success before writing settings.

In [ ]:
backend_outputs = None
print("Deploying backend infrastructure...")
backend_deployment = resource_client.deployments.begin_create_or_update(
    BACKEND_RESOURCE_GROUP,
    BACKEND_DEPLOYMENT_NAME,
    {
        "properties": {
            "mode": "Incremental",
            "template": backend_template,
            "parameters": {
                "demoId": {"value": DEMO_ID},
                "location": {"value": BACKEND_LOCATION},
                "modelVersion": {"value": model_version},
                "deploymentName": {"value": MODEL_DEPLOYMENT_NAME},
                "modelCapacity": {"value": MODEL_CAPACITY},
            },
        }
    },
).result()
backend_outputs = backend_deployment.properties.outputs
print("Backend deployment completed")
print(json.dumps(backend_outputs, indent=2))

## Write env/.env

Both deployments must have completed successfully. A failure must not overwrite the previous successful settings.

In [ ]:
env_values = {
    "AZURE_SUBSCRIPTION_ID": outputs["subscriptionId"]["value"],
    "AZURE_RESOURCE_GROUP": outputs["resourceGroupName"]["value"],
    "AZURE_LOCATION": outputs["location"]["value"],
    "AZURE_ACCOUNT_NAME": outputs["foundryAccountName"]["value"],
    "AZURE_AI_RESOURCE_ID": outputs["foundryResourceId"]["value"],
    "AZURE_AI_ENDPOINT": outputs["foundryEndpoint"]["value"],
    "AZURE_AI_PROJECT_NAME": outputs["projectName"]["value"],
    "AZURE_AI_PROJECT_ID": outputs["projectResourceId"]["value"],
    "AZURE_AI_PROJECT_ENDPOINT": outputs["projectEndpoint"]["value"],
    "AZURE_APIM_NAME": outputs["apimServiceName"]["value"],
    "AZURE_APIM_RESOURCE_ID": outputs["apimResourceId"]["value"],
    "APIM_ENDPOINT": outputs["apimGatewayUrl"]["value"],
    "AZURE_SEARCH_NAME": outputs["searchServiceName"]["value"],
    "AZURE_SEARCH_RESOURCE_ID": outputs["searchResourceId"]["value"],
    "AZURE_SEARCH_ENDPOINT": outputs["searchEndpoint"]["value"],
    "AZURE_BACKEND_RESOURCE_GROUP": backend_outputs["backendResourceGroupName"]["value"],
    "AZURE_BACKEND_LOCATION": backend_outputs["backendLocation"]["value"],
    "AZURE_BACKEND_ACCOUNT_NAME": backend_outputs["backendAccountName"]["value"],
    "AZURE_BACKEND_RESOURCE_ID": backend_outputs["backendResourceId"]["value"],
    "AZURE_BACKEND_ENDPOINT": backend_outputs["backendEndpoint"]["value"],
    "AZURE_BACKEND_OPENAI_ENDPOINT": backend_outputs["backendOpenAIEndpoint"]["value"],
    "AZURE_BACKEND_DEPLOYMENT_NAME": backend_outputs["backendDeploymentName"]["value"],
    "AZURE_BACKEND_MODEL_VERSION": backend_outputs["backendModelVersion"]["value"],
    "AZURE_BACKEND_MODEL_SKU": backend_outputs["backendModelSku"]["value"],
    "AZURE_BACKEND_MODEL_CAPACITY": backend_outputs["backendModelCapacity"]["value"],
}
generated_ts = datetime.now(timezone.utc).isoformat()
env_content = (
    "# BYOM capability testing - generated by 01_deploy_infra.ipynb\n"
    f"# Generated UTC: {generated_ts}\n\n"
    + "\n".join(f"{key}={value}" for key, value in env_values.items())
    + "\n"
)
env_path.write_text(env_content, encoding="utf-8")
print(f"Wrote {env_path}")

## Summary

The consumer foundation and separate Sweden Central GPT-5.4 backend are provisioned. APIM has a system-assigned identity. `env/.env` contains both sets of identifiers/endpoints. Continue with [02_configure.ipynb](02_configure.ipynb) to configure the gateway and connected model, then [03_test.ipynb](03_test.ipynb). Foundry IQ connections remain out of scope.

When finished, follow the README cleanup instructions for both groups: target the backend in a fresh kernel, run the unchanged destroy notebook, then repeat for the consumer group and purge its APIM service.